# BÁO CÁO THỰC NGHIỆM PHÂN CỤM KHÁCH HÀNG (RFM & K-MEANS)
## Môn học: Kho dữ liệu và Khai phá dữ liệu - PTIT
**Mục tiêu:** Xây dựng đặc trưng Recency - Frequency - Monetary, áp dụng biến đổi $\ln(1+x)$ xử lý phân phối lệch phải, chuẩn hóa StandardScaler, tìm số cụm tối ưu $K$ bằng phương pháp Elbow và Silhouette Score, và diễn giải hồ sơ cụm phục vụ chăm sóc khách hàng.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sys.path.append("..")
from src.mining.rfm_extractor import extract_rfm_from_dwh
from src.mining.clustering import preprocess_rfm, assign_cluster_names
from src.mining.evaluation import evaluate_k_selection

# 1. Trích xuất đặc trưng RFM
rfm_df, ref_date = extract_rfm_from_dwh(source_code="UCI_HISTORICAL")
print(f"Ngày tham chiếu T = {ref_date}")
rfm_df.head()

### 1. Phân tích Phân phối Đặc trưng RFM
Trong thực tế bán lẻ, phần lớn khách hàng chỉ mua 1-2 lần với giá trị nhỏ, trong khi một số ít khách VIP/mua buôn chi tiêu rất lớn. Điều này tạo ra phân phối lệch phải (Right-skewed) cực kỳ nghiêm trọng.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(rfm_df['recency'], kde=True, ax=axes[0], color='blue')
axes[0].set_title('Phân phối Recency (Ngày)')

sns.histplot(rfm_df['frequency'], kde=True, ax=axes[1], color='green')
axes[1].set_title('Phân phối Frequency (Số hóa đơn)')

sns.histplot(rfm_df['monetary'], kde=True, ax=axes[2], color='purple')
axes[2].set_title('Phân phối Monetary (Tổng chi tiêu)')
plt.tight_layout()
plt.show()

### 2. Tiền xử lý: Biến đổi $\ln(1+x)$ và Chuẩn hóa StandardScaler
K-means dựa trên khoảng cách Euclidean nên rất nhạy với thang đo và ngoại lệ. Biến đổi log kéo các giá trị ngoại lệ về gần tâm phân phối, giúp thuật toán gom cụm chính xác hơn.

In [ ]:
X_scaled, scaler = preprocess_rfm(rfm_df)
scaled_df = pd.DataFrame(X_scaled, columns=['R_scaled', 'F_scaled', 'M_scaled'])
scaled_df.describe()

### 3. Thực nghiệm Tìm $K$ Tối ưu (Elbow Method & Silhouette Score)

In [ ]:
eval_results = evaluate_k_selection(rfm_df, k_range=range(2, 7))
best_k = eval_results['recommended_k']
print(f"Số cụm được khuyến nghị theo Silhouette: K = {best_k}")

### 4. Huấn luyện Mô hình K-means & Gán nhãn Hồ sơ Cụm (Profiling)

In [ ]:
kmeans = KMeans(n_clusters=best_k, init='k-means++', n_init=20, random_state=42)
rfm_df['cluster_id'] = kmeans.fit_predict(X_scaled)
cluster_names = assign_cluster_names(rfm_df)
rfm_df['cluster_name'] = rfm_df['cluster_id'].map(cluster_names)

# Tổng hợp hồ sơ cụm
profile = rfm_df.groupby('cluster_name').agg({
    'customer_key': 'count',
    'recency': ['median', 'mean'],
    'frequency': ['median', 'mean'],
    'monetary': ['median', 'sum']
})
profile

### 5. Đề xuất Chiến lược Chăm sóc Khách hàng Theo Cụm
1. **VIP / Champions:** Đóng góp tỷ trọng doanh thu vượt trội $\to$ Chăm sóc 1-1, quà tri ân đặc quyền, quyền mua hàng trước.
2. **Khách hàng Trung thành (Loyal):** Tần suất mua cao $\to$ Tích điểm đổi quà, gợi ý bán kèm sản phẩm (Cross-sell).
3. **Khách hàng Tiềm năng (Potential):** Mới mua gần đây $\to$ Gửi voucher giảm giá lần 2 để kích thích tái mua hàng.
4. **Khách hàng Nguy cơ rời bỏ (At Risk):** Từng chi tiêu nhiều nhưng lâu rồi chưa quay lại $\to$ Chiến dịch email/SMS kích cầu cá nhân hóa.